## задание 1 

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score
from catboost import CatBoostClassifier
import numpy as np
from sentence_transformers import SentenceTransformer, util
import requests

pd.set_option('display.max_columns', 100)

ratings = pd.read_csv('ratings.csv')
ratings

C:\Users\User\AppData\Local\Programs\Python\Python310\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3
...,...,...,...
5976474,49925,510,5
5976475,49925,528,4
5976476,49925,722,4
5976477,49925,949,5


In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df, train_y, test_y = train_test_split(ratings[['user_id', 'book_id']], ratings.rating, train_size=0.8)

books = pd.read_csv('books.csv')



list(books)

genres_df = pd.read_json('goodreads_book_genres_initial.json', lines=True)

genres_df = genres_df[genres_df.book_id.isin(books.goodreads_book_id)]

genres_df.columns = ['book_id', 'genres_dict']



all_genres = set()
for dict_genre in genres_df.genres_dict:
    for elem in list(dict_genre.keys()):
        all_genres.add(elem)

all_genres

for genre in all_genres:
    genres_df[genre] = 0

def simple_one_hot(genre_dict, genre):
    if genre in genre_dict:
        return 1
    return 0

for genre in all_genres:
    genres_df[genre] = genres_df.apply(lambda df: simple_one_hot(df['genres_dict'], genre), axis=1)



train_df = train_df.merge(books[['book_id', 'goodreads_book_id']], left_on='book_id', right_on='book_id', how='left')
test_df = test_df.merge(books[['book_id', 'goodreads_book_id']], left_on='book_id', right_on='book_id', how='left')

train_df = train_df.merge(genres_df, left_on='goodreads_book_id', right_on='book_id', how='left')
test_df = test_df.merge(genres_df, left_on='goodreads_book_id', right_on='book_id', how='left')

train_df.head(5)

users_profiles = train_df.groupby('user_id')[list(all_genres)].sum()

users_profiles.columns = ['user_'+name for name in list(users_profiles)]


train_df = train_df.merge(users_profiles, left_on='user_id', right_on='user_id', how='left')
test_df = test_df.merge(users_profiles, left_on='user_id', right_on='user_id', how='left')

train_df.columns = ['book_'+item if item in all_genres else item for item in list(train_df)]
test_df.columns = ['book_'+item if item in all_genres else item for item in list(test_df)]


from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=100, stop_words=stopwords.words('english'))
X = vectorizer.fit_transform(books.title)

train_df = train_df.merge(users_profiles, left_on='user_id', right_on='user_id', how='left')
test_df = test_df.merge(users_profiles, left_on='user_id', right_on='user_id', how='left')

tf_idf_df = pd.DataFrame(X.toarray(), columns=vectorizer.vocabulary_)

books = pd.concat([books, tf_idf_df], axis=1)

train_df = train_df.merge(books[list(vectorizer.vocabulary_)+['book_id']], left_on='book_id_x', right_on='book_id')
test_df = test_df.merge(books[list(vectorizer.vocabulary_)+['book_id']], left_on='book_id_x', right_on='book_id')

# 1. Признак для пользователя (средний рейтинг)
user_avg_ratings = ratings.groupby('user_id')['rating'].mean().reset_index()
user_avg_ratings = user_avg_ratings.rename(columns={'rating': 'user_avg_rating'})
train_df = train_df.merge(user_avg_ratings, on='user_id', how='left')
test_df = test_df.merge(user_avg_ratings, on='user_id', how='left')

# 2. Признак для книги (популярность)
books['book_popularity'] = np.log(books['ratings_count'] + 1) # +1 чтобы избежать логарифма от нуля
train_df = train_df.merge(books[['book_id', 'book_popularity']], on='book_id', how='left')
test_df = test_df.merge(books[['book_id', 'book_popularity']], on='book_id', how='left')

'''
1. Признак для пользователя:
user_avg_rating: Средний рейтинг пользователя.
Обоснование: Этот признак отражает общий уровень предпочтений пользователя: любит ли он книги, которые ему нравятся, 
или предпочитает средние оценки.

2. Признак для книги:
book_popularity: Популярность книги, выраженная как логарифм количества оценок (ratings_count).
Обоснование: Логарифмирование количества оценок сглаживает распределение данных, так как очень популярные книги могут 
иметь значительно большее количество оценок, чем менее популярные.
'''

cols_to_drop = ['user_id', 'book_id_x', 'goodreads_book_id', 'book_id_y', 'genres_dict']

# Обучение модели
CB_model = CatBoostClassifier(
                            iterations=1000,
                            depth=6,
                            learning_rate=0.1,
                            task_type="GPU",
                            loss_function='MultiClass',
                            use_best_model=True,
                            eval_metric='Accuracy',  
                            early_stopping_rounds=10,
                            custom_metric='Precision'  
                        )

CB_model.fit(train_df.drop(columns=cols_to_drop), 
              train_y, 
              eval_set=(test_df.drop(columns=cols_to_drop), test_y),
              verbose=False,
              plot=True) 